# Scholarly Metadata Review Protocol Notebook

This public notebook is a lightweight template for running a metadata-only literature review workflow with the local app. It intentionally avoids project-specific search terms, reviewer decisions, private exports, and unpublished screening results.

Use it to document the protocol shape:

1. Load app exports or the local SQLite metadata database.
2. Define the review question and source plan.
3. Run a preliminary exploratory search on a small set of sources.
4. Convert observed terms into search pillars.
5. Track source yields and deduplication.
6. Export a human screening workbook.
7. Produce protocol-ready summary tables.

No full-text scraping or PDF downloading is performed.

## 0. Setup

Run the local web app first, export a result set, then open this notebook from the app's **Notebook tools** panel. If the app writes `notebooks/analysis_context.json`, this notebook will use that export automatically. Otherwise, it falls back to the local SQLite database in `data/metadata.sqlite`.

In [ ]:
from pathlib import Path
import json
import sqlite3
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

REPORT_DIR = ROOT / "reports" / "public_protocol_template"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

context_path = ROOT / "notebooks" / "analysis_context.json"
context = json.loads(context_path.read_text()) if context_path.exists() else {}
context

## 1. Load Metadata Records

The expected unit is one metadata record per paper or citation. Useful columns include `title`, `abstract`, `author_keywords`, `indexed_keywords`, `authors`, `year`, `doi`, `journal`, `source_databases`, `source_urls`, and duplicate/status fields.

In [ ]:
def load_metadata_frame():
    export_path = Path(context.get("export_path", "")) if context.get("export_path") else None
    if export_path and export_path.exists():
        if export_path.suffix.lower() == ".xlsx":
            return pd.read_excel(export_path)
        if export_path.suffix.lower() == ".csv":
            return pd.read_csv(export_path)
        if export_path.suffix.lower() == ".jsonl":
            return pd.read_json(export_path, lines=True)
        raise ValueError(f"Unsupported export type: {export_path.suffix}")

    db_path = ROOT / "data" / "metadata.sqlite"
    if db_path.exists():
        with sqlite3.connect(db_path) as conn:
            return pd.read_sql_query("select * from records", conn)

    return pd.DataFrame()

df = load_metadata_frame()
print(f"Loaded {len(df):,} metadata records")
df.head()

## 2. Protocol Definition

Edit this table for your review before running final searches. The example values are placeholders, not recommended search terms.

In [ ]:
protocol = {
    "review_title": "<working review title>",
    "review_type": "scoping review",
    "metadata_only": True,
    "full_text_downloads": False,
    "population_or_concept": "<Concept A>",
    "method_or_exposure": "<Concept B>",
    "optional_task_or_outcome": "<Concept C, optional>",
    "date_prepared": date.today().isoformat(),
}

pd.DataFrame(protocol.items(), columns=["protocol_item", "value"])

## 3. Source Plan

Use open metadata sources for routine local harvesting. Use licensed databases only when you have credentials or when you manually export citation files from the database interface.

In [ ]:
source_plan = pd.DataFrame([
    {"source": "PubMed / MEDLINE", "mode": "API", "credential": "optional NCBI_API_KEY", "notes": "Use database-specific PubMed syntax."},
    {"source": "Europe PMC", "mode": "API", "credential": "none", "notes": "Useful biomedical supplement."},
    {"source": "OpenAlex", "mode": "API", "credential": "optional OPENALEX_API_KEY", "notes": "Broad scholarly metadata."},
    {"source": "Crossref", "mode": "API", "credential": "none", "notes": "DOI and publisher metadata."},
    {"source": "Semantic Scholar", "mode": "API", "credential": "optional SEMANTIC_SCHOLAR_API_KEY", "notes": "Bulk metadata endpoint with pagination."},
    {"source": "DOAJ", "mode": "API", "credential": "none", "notes": "Open access journal metadata."},
    {"source": "CORE", "mode": "API", "credential": "CORE_API_KEY", "notes": "Use only when configured."},
    {"source": "Scopus", "mode": "licensed API or manual export", "credential": "ELSEVIER_API_KEY", "notes": "Requires entitlement."},
    {"source": "Web of Science", "mode": "manual export", "credential": "licensed access", "notes": "Import RIS/BibTeX/CSV exports."},
    {"source": "IEEE Xplore", "mode": "API or manual export", "credential": "IEEE_API_KEY", "notes": "Observe key-specific quotas."},
    {"source": "ACM Digital Library", "mode": "manual export", "credential": "licensed/public portal", "notes": "Import exported citations."},
])
source_plan

## 4. Preliminary Exploratory Search

Before final comprehensive searching, run a limited exploratory search on two high-value sources. The purpose is not comprehensiveness. The purpose is to discover natural title words, abstract words, author keywords, and index terms that should inform the final search strings.

Recommended pattern:

- Source 1: a domain-specific bibliographic database.
- Source 2: a broad multidisciplinary or licensed index.
- Record target: approximately 10-20 relevant-looking records.
- Output: candidate terms for each search pillar.

In [ ]:
source_col = "source_databases" if "source_databases" in df.columns else None
if source_col and not df.empty:
    exploratory_df = df[df[source_col].astype(str).str.contains("pubmed|scopus|openalex|semantic", case=False, na=False)].copy()
    if exploratory_df.empty:
        exploratory_df = df.copy()
else:
    exploratory_df = df.copy()

exploratory_columns = [col for col in ["title", "abstract", "author_keywords", "indexed_keywords", "year", "doi", source_col] if col and col in exploratory_df.columns]
exploratory_df[exploratory_columns].head(20) if exploratory_columns else exploratory_df.head(20)

## 5. Search Pillar Template

The protocol uses three editable pillars:

- **Pillar 1:** condition, population, or core concept.
- **Pillar 2:** method, intervention, exposure, or technology.
- **Pillar 3:** task, outcome, role, or context. This is optional and often best reserved for sensitivity searches or targeted gap checks.

In [ ]:
pillar_table = pd.DataFrame([
    {"pillar": "Pillar 1", "concept": "Condition / population / concept", "term": "<primary concept term>", "role": "core"},
    {"pillar": "Pillar 1", "concept": "Condition / population / concept", "term": "<synonym or controlled vocabulary term>", "role": "core or optional"},
    {"pillar": "Pillar 2", "concept": "Method / intervention / exposure", "term": "<primary method term>", "role": "core"},
    {"pillar": "Pillar 2", "concept": "Method / intervention / exposure", "term": "<synonym or narrower method term>", "role": "core or optional"},
    {"pillar": "Pillar 3", "concept": "Task / outcome / context", "term": "<optional task or outcome term>", "role": "targeted sweep"},
])

pillar_path = REPORT_DIR / "search_pillar_template.csv"
pillar_table.to_csv(pillar_path, index=False)
pillar_table

## 6. Draft Database-Specific Search Strings

Use Pillar 1 + Pillar 2 as the broad base search. Add Pillar 3 only for targeted supplementary sweeps or when a database returns an unmanageably large result set.

In [ ]:
def quoted_or_block(terms):
    clean_terms = [str(term).strip() for term in terms if str(term).strip() and not str(term).startswith("<")]
    return " OR ".join(f'"{term}"' for term in clean_terms) if clean_terms else '"<term 1>" OR "<term 2>"'

p1 = quoted_or_block(pillar_table.loc[pillar_table["pillar"].eq("Pillar 1"), "term"])
p2 = quoted_or_block(pillar_table.loc[pillar_table["pillar"].eq("Pillar 2"), "term"])
p3 = quoted_or_block(pillar_table.loc[pillar_table["pillar"].eq("Pillar 3"), "term"])

search_string_templates = pd.DataFrame([
    {"database": "PubMed / MEDLINE", "core_string": f"(({p1}) AND ({p2}))", "optional_task_add_on": f"AND ({p3})", "note": "Add field tags such as [TIAB] and controlled vocabulary where appropriate."},
    {"database": "Scopus", "core_string": f"TITLE-ABS-KEY(({p1}) AND ({p2}))", "optional_task_add_on": f"AND TITLE-ABS-KEY({p3})", "note": "Requires licensed API/portal access."},
    {"database": "Web of Science", "core_string": f"TS=(({p1}) AND ({p2}))", "optional_task_add_on": f"AND TS=({p3})", "note": "Usually manual export/import unless API entitlement is configured."},
    {"database": "Open metadata APIs", "core_string": f"({p1}) AND ({p2})", "optional_task_add_on": f"AND ({p3})", "note": "Use compact Boolean strings for Europe PMC, OpenAlex, Crossref, Semantic Scholar, DOAJ, and CORE."},
])

search_path = REPORT_DIR / "database_search_string_templates.xlsx"
search_string_templates.to_excel(search_path, index=False)
search_string_templates

## 7. Identification and Deduplication Summary

This table gives a lightweight identification summary from loaded metadata. Treat it as a working log, not final PRISMA reporting, until human screening is complete.

In [ ]:
def split_sources(value):
    if isinstance(value, list):
        return value
    return [item.strip() for item in str(value).replace(",", ";").split(";") if item.strip()]

if not df.empty and "source_databases" in df.columns:
    source_counts = (
        df["source_databases"]
        .fillna("")
        .map(split_sources)
        .explode()
        .replace("", pd.NA)
        .dropna()
        .value_counts()
        .rename_axis("source")
        .reset_index(name="raw_records")
    )
else:
    source_counts = pd.DataFrame(columns=["source", "raw_records"])

if not df.empty:
    dedupe_basis = "doi" if "doi" in df.columns else "title"
    unique_estimate = df[dedupe_basis].fillna("").astype(str).str.lower().str.strip().replace("", pd.NA).nunique()
else:
    dedupe_basis = "not available"
    unique_estimate = 0

summary = pd.DataFrame([
    {"stage": "Identification", "metric": "Raw loaded records", "count": len(df), "note": "From current export or local database."},
    {"stage": "Deduplication", "metric": f"Approximate unique records by {dedupe_basis}", "count": unique_estimate, "note": "Use app deduplication for final canonical counts."},
])

source_counts, summary

## 8. Human Screening Workbook

Export a neutral screening workbook that can be filled by reviewers. Keep completed reviewer decisions private unless you intentionally anonymize and publish them.

In [ ]:
screening_columns = [col for col in ["title", "abstract", "authors", "year", "journal", "doi", "source_databases", "source_urls"] if col in df.columns]
if df.empty:
    screening_df = pd.DataFrame(columns=screening_columns)
else:
    screening_df = df[screening_columns].copy()

for col in [
    "reviewed_status",
    "title_abstract_decision",
    "exclusion_reason",
    "full_text_needed",
    "final_inclusion_decision",
    "reviewer_initials",
    "review_notes",
]:
    screening_df[col] = ""

screening_path = REPORT_DIR / "human_screening_template.xlsx"
with pd.ExcelWriter(screening_path, engine="openpyxl") as writer:
    screening_df.to_excel(writer, sheet_name="screening", index=False)
    pd.DataFrame([
        {"field": "reviewed_status", "allowed_values": "not reviewed; in progress; complete"},
        {"field": "title_abstract_decision", "allowed_values": "include; exclude; maybe"},
        {"field": "exclusion_reason", "allowed_values": "wrong population/concept; wrong method; non-research item; duplicate; insufficient metadata; other"},
        {"field": "final_inclusion_decision", "allowed_values": "include; exclude; pending"},
    ]).to_excel(writer, sheet_name="instructions", index=False)

screening_path

## 9. Protocol Output Bundle

This cell writes reusable public/protocol artifacts only: source plan, search-pillar template, search-string templates, and identification summary. It does not write API keys, private data, or completed screening decisions.

In [ ]:
source_plan.to_csv(REPORT_DIR / "source_plan.csv", index=False)
summary.to_csv(REPORT_DIR / "identification_summary_template.csv", index=False)
if not source_counts.empty:
    source_counts.to_csv(REPORT_DIR / "source_yield_summary.csv", index=False)

print("Wrote protocol template outputs to:", REPORT_DIR)

## 10. Methods Text Skeleton

Adapt this text in your protocol or manuscript. Replace placeholders with your topic, databases, dates, and final counts.

In [ ]:
methods_text = (
    "Searches were developed iteratively using a preliminary exploratory phase followed by database-specific final searches. "
    "The exploratory phase examined a limited set of metadata records to identify natural-language terms, controlled vocabulary, "
    "author keywords, and indexed terms. Final search strings combined terms from two core pillars: the review concept/population "
    "and the method/intervention/exposure. A third optional task/outcome pillar was reserved for targeted supplementary searches "
    "and sensitivity checks. Records were collected as metadata only, deduplicated using persistent identifiers and title/year "
    "matching, and exported to a human screening workbook for title/abstract review. No full-text scraping or PDF downloading was performed."
)

print(methods_text)